# 04 — Pattern Structure Validation

**Author:** Zeineb Turki  
**Date:** April 2026  
**Phase:** 3.5 — Structure Validation  

## Objective

The triangle and channel detectors now export trendline coefficients via `return_details=True`.
This notebook visually validates whether detected patterns match real chart structures by
overlaying fitted upper/lower boundary lines on candlestick charts.

**Why this matters (supervisor feedback):**  
Exporting trendline structure proves that detections are geometrically correct — not just
statistical artifacts. A triangle should show converging lines; a channel should show
parallel lines with price bouncing between them.

**Scope:**
- Overlay upper (resistance) and lower (support) trendlines on detected patterns
- Grid views: 6 triangles, 6 channels with boundary lines
- Zoomed individual examples for presentation
- Summary statistics by pattern type and time

In [ ]:
import sys, os
from pathlib import Path

if 'google.colab' in str(getattr(sys, 'modules', {})) or os.path.exists('/content'):
    REPO_DIR  = '/content/regime-aware-ml-trading'
    PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
    if not os.path.isdir(PROJ_ROOT):
        os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')
    os.system(f'{sys.executable} -m pip install -q hmmlearn scikit-learn seaborn statsmodels')
else:
    # Find project root by looking for src/ directory
    def _find_project_root():
        current = Path.cwd()
        for _ in range(10):  # Prevent infinite loop
            if (current / "src").is_dir():
                return current
            current = current.parent
        # Fallback: assume we're in notebooks/ subdirectory
        return Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
    
    PROJ_ROOT = str(_find_project_root())

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.data.load_data import load_spy
from src.patterns.export_patterns import collect_pattern_details
from src.utils.plotting import plot_candlestick, add_trendline, add_event_marker

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Load data and collect pattern details
df = load_spy()
all_details = collect_pattern_details(df)

# Split by pattern family
tri_details = [d for d in all_details if 'triangle' in d['pattern_type']]
ch_details  = [d for d in all_details if 'channel' in d['pattern_type']]

print(f'Loaded {len(df)} bars: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Triangles: {len(tri_details)}  |  Channels: {len(ch_details)}  |  Total: {len(all_details)}')

## 1. Triangle Pattern Overview

The triangle detector identifies four signal types:

- **Ascending triangle** — flat upper trendline (resistance) + rising lower trendline (support). Breakout above.
- **Descending triangle** — falling upper trendline + flat lower trendline. Breakout below.
- **Symmetric triangle** — both trendlines converge (falling highs, rising lows). Breakout either direction.
- **Desc. triangle upper test** — within a descending triangle, price approaches the falling resistance line. A discretionary setup signal.

Each detection includes `upper_slope`, `upper_intercept`, `lower_slope`, `lower_intercept` from linear regression on the window's highs/lows.

In [ ]:
# Helper to plot a single detection on a given axes
FORWARD_CONTEXT = 10

def plot_detection(ax, det, df):
    """Draw candlestick + trendlines + event marker for one detection."""
    start = det['start_idx']
    end = min(det['end_idx'] + FORWARD_CONTEXT, len(df) - 1)
    chart_slice = df.iloc[start:end + 1]
    window_slice = df.iloc[det['start_idx']:det['end_idx']]

    plot_candlestick(chart_slice, ax=ax,
                     title=f"{det['pattern_type']}\n{det['event_date'].strftime('%Y-%m-%d')}")

    upper_coeffs = [det['upper_slope'], det['upper_intercept']]
    lower_coeffs = [det['lower_slope'], det['lower_intercept']]

    add_trendline(ax, window_slice, upper_coeffs, det['window'],
                  color='red', label='Upper')
    add_trendline(ax, window_slice, lower_coeffs, det['window'],
                  color='blue', label='Lower')

    event_price = df.loc[det['event_date'], 'Close']
    add_event_marker(ax, det['event_date'], event_price,
                     marker='v', color='orange', size=100, label='Detection')


# Select 6 representative triangles (spread across types)
tri_by_type = {}
for d in tri_details:
    tri_by_type.setdefault(d['pattern_type'], []).append(d)

selected_tri = []
for ptype, dets in sorted(tri_by_type.items()):
    # Take up to 2 per type, spread evenly
    step = max(1, len(dets) // 2)
    selected_tri.extend(dets[::step][:2])
selected_tri = selected_tri[:6]

print(f'Selected {len(selected_tri)} triangles for grid:')
for d in selected_tri:
    print(f"  {d['pattern_type']:30s}  {d['event_date'].strftime('%Y-%m-%d')}")

# Plot 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, ax in enumerate(axes.flat):
    if idx < len(selected_tri):
        plot_detection(ax, selected_tri[idx], df)
        ax.legend(fontsize=7, loc='upper left')
    else:
        ax.set_visible(False)

fig.suptitle('Triangle Detections — Trendline Overlay', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Triangle Observations

- **Convergence is visible:** upper (red) and lower (blue) lines clearly narrow over the window
- **Breakout placement:** the orange marker sits at or beyond the formation boundary, confirming the exit signal
- **Type classification matches geometry:** ascending triangles show flat tops with rising bottoms, symmetric show both converging
- **Upper test signals:** for descending triangles, the detection fires when price approaches falling resistance — a discretionary setup

## 2. Channel Pattern Overview

The channel detector identifies two types:

- **Channel up** — both trendlines slope upward with parallel structure. Signal fires when price touches a boundary.
- **Channel down** — both trendlines slope downward. Same boundary-touch trigger.

Valid channels require R² >= 0.70 on both regressions, at least 4 distinct touches per band (5-bar separation), and width between 1-6x ATR.

In [ ]:
# Select 6 representative channels (spread across types)
ch_by_type = {}
for d in ch_details:
    ch_by_type.setdefault(d['pattern_type'], []).append(d)

selected_ch = []
for ptype, dets in sorted(ch_by_type.items()):
    step = max(1, len(dets) // 3)
    selected_ch.extend(dets[::step][:3])
selected_ch = selected_ch[:6]

print(f'Selected {len(selected_ch)} channels for grid:')
for d in selected_ch:
    print(f"  {d['pattern_type']:30s}  {d['event_date'].strftime('%Y-%m-%d')}")

# Plot 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, ax in enumerate(axes.flat):
    if idx < len(selected_ch):
        plot_detection(ax, selected_ch[idx], df)
        ax.legend(fontsize=7, loc='upper left')
    else:
        ax.set_visible(False)

fig.suptitle('Channel Detections — Trendline Overlay', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Channel Observations

- **Parallelism confirmed:** upper and lower lines maintain consistent spacing across the window
- **Price respects boundaries:** candlesticks bounce between the two trendlines with visible touch points
- **Width is meaningful:** channels are wide enough to trade but not so wide they lose structure
- **R² filtering works:** the 0.70 threshold eliminates weak fits — remaining channels show clear linear structure

In [ ]:
# Zoomed individual examples: 1 triangle + 1 channel
# Pick the middle detection from each list for a representative example
zoom_tri = tri_details[len(tri_details) // 2]
zoom_ch  = ch_details[len(ch_details) // 2]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Triangle zoom
plot_detection(axes[0], zoom_tri, df)
axes[0].legend(fontsize=9)
axes[0].set_title(f"Triangle: {zoom_tri['pattern_type']}\n{zoom_tri['event_date'].strftime('%Y-%m-%d')}",
                  fontsize=11)

# Channel zoom
plot_detection(axes[1], zoom_ch, df)
axes[1].legend(fontsize=9)
axes[1].set_title(f"Channel: {zoom_ch['pattern_type']}\n{zoom_ch['event_date'].strftime('%Y-%m-%d')}",
                  fontsize=11)

fig.suptitle('Zoomed Examples — Triangle vs Channel', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
details_df = pd.DataFrame(all_details)
details_df['year'] = details_df['event_date'].dt.year

# --- Count by pattern type ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_counts = details_df['pattern_type'].value_counts().sort_index()
type_counts.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_xlabel('Count')
axes[0].set_title('Detections by Pattern Type')
for i, v in enumerate(type_counts.values):
    axes[0].text(v + 0.3, i, str(v), va='center', fontsize=9)

# --- Distribution over time (yearly) ---
yearly = details_df.groupby(['year', 'pattern_type']).size().unstack(fill_value=0)
yearly.plot.bar(ax=axes[1], stacked=True, colormap='tab10')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Count')
axes[1].set_title('Detections by Year')
axes[1].legend(fontsize=7, loc='upper left')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# --- Summary table ---
summary = details_df.groupby('pattern_type').agg(
    count=('event_date', 'size'),
    first=('event_date', 'min'),
    last=('event_date', 'max'),
    avg_upper_slope=('upper_slope', 'mean'),
    avg_lower_slope=('lower_slope', 'mean'),
)
summary['first'] = summary['first'].dt.strftime('%Y-%m-%d')
summary['last']  = summary['last'].dt.strftime('%Y-%m-%d')

print('\n=== Pattern Detection Summary ===')
print(f'Total detections: {len(all_details)}')
print(f'  Triangles: {len(tri_details)}')
print(f'  Channels:  {len(ch_details)}')
print(f'  Date range: {details_df["event_date"].min().strftime("%Y-%m-%d")} to '
      f'{details_df["event_date"].max().strftime("%Y-%m-%d")}')
print()
display(summary)

---

## Conclusion & Next Steps

**Structure validation results:**

| Metric | Value |
|--------|-------|
| Total patterns with trendline structure | See summary above |
| Triangle detections | Converging upper/lower lines confirmed |
| Channel detections | Parallel boundaries with price respecting bands |
| Trendline overlay | All detections have valid slope/intercept coefficients |

**Key findings:**
- Visual inspection confirms that `return_details=True` produces geometrically correct trendlines
- Triangle convergence is visible in the charts — lines narrow as expected
- Channel parallelism holds — the R² >= 0.70 filter keeps only well-fitted structures
- Event markers (orange) sit at the correct breakout or boundary-touch location
- The exported coefficients (slope + intercept) can be used downstream as features

**Next step:** `05_hmm_regimes.ipynb` — Fit HMM market regimes using daily log returns and rolling volatility. Regimes will later be joined to the event dataset as context for ML models.